### this is a development environment for decoding code. Once it works here, throw it into the main function in decoding_dcc.py.

set up imports and testing variables (all lowercase - note these are the variables from run_decoding_dcc.py but defined here rather than imported)

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import json
import pickle
from functools import partial
from datetime import datetime
from types import SimpleNamespace

import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from scipy.stats import ttest_ind, ttest_rel
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

from ieeg.navigate import channel_outlier_marker, trial_ieeg, crop_empty_data, outliers_to_nan
from ieeg.io import raw_from_layout, get_data
from ieeg.calc.stats import time_perm_cluster
from ieeg.calc.fast import mean_diff
from ieeg.decoding.models import PcaLdaClassification
from ieeg.calc.oversample import MinimumNaNSplit

try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.analysis.config import experiment_conditions
from src.analysis.config.condition_registry import get_comparisons, get_pooled_shuffle_settings, get_conditions_obj
from src.analysis.utils.general_utils import (
    make_or_load_subjects_electrodes_to_ROIs_dict,
    load_subjects_electrodes_to_ROIs_dict,
    identify_bad_channels_by_trial_nan_rate,
    impute_trial_nans_by_channel_mean,
    create_subjects_mne_objects_dict,
    filter_electrode_lists_against_subjects_mne_objects,
    find_difference_between_two_electrode_lists,
    print_summary_of_dropped_electrodes,
    get_conditions_save_name,
    get_default_LAB_root,
    get_sig_chans_per_subject,
    make_sig_electrodes_per_subject_and_roi_dict,
)

from src.analysis.utils.labeled_array_utils import (
    put_data_in_labeled_array_per_roi_subject,
    remove_nans_from_labeled_array,
    remove_nans_from_all_roi_labeled_arrays,
    concatenate_conditions_by_string,
    get_data_in_time_range,
    make_bootstrapped_roi_labeled_arrays_with_nan_trials_removed_for_each_channel,
    gather_class_data_by_stratum
)

from src.analysis.decoding.decoding import (
    get_and_plot_confusion_matrix_for_rois_jim,
    Decoder,
    windower,
    get_confusion_matrices_for_rois_time_window_decoding_jim,
    compute_accuracies,
    plot_true_vs_shuffle_accuracies,
    plot_accuracies_with_multiple_sig_clusters,
    plot_accuracies_nature_style,
    make_pooled_shuffle_distribution,
    find_significant_clusters_of_series_vs_distribution_based_on_percentile,
    compute_pooled_bootstrap_statistics,
    do_time_perm_cluster_comparing_two_true_bootstrap_accuracy_distributions,
    do_mne_paired_cluster_test,
    get_pooled_accuracy_distributions_for_comparison,
    get_time_averaged_confusion_matrix,
    cluster_perm_paired_ttest_by_duration,
    run_two_one_tailed_tests_with_time_perm_cluster,
    extract_pooled_cm_traces,
    plot_cm_traces_nature_style,
    run_context_comparison_analysis,
    plot_cross_block_overlay,
    concatenate_and_balance_data_for_decoding,
    plot_static_pca_projection,
    plot_pca_over_time,
    plot_pca_3d_trajectory,
    plot_static_umap_projection,
    plot_umap_3d_trajectory
)

from src.analysis.decoding.process_bootstrap import process_bootstrap
from src.analysis.decoding.run_visualization_debug import run_visualization_debug
from src.analysis.decoding.run_debug_cm_traces import run_debug_cm_traces
from src.analysis.decoding.run_aggregate_and_plot_time_averaged_cms import run_aggregate_and_plot_time_averaged_cms
from src.analysis.decoding.run_context_comparisons import run_all_context_comparisons

# task
task = 'GlobalLocal'

# trial selection
# switched to False for err-corr decoding
acc_trials_only = True

# decoding parameters
# first, choose your classifier
model_choice = 'LDA'  # options: 'LDA', 'SVC'

if model_choice == 'LDA':
    clf_model = LinearDiscriminantAnalysis()
    clf_model_str = 'LDA'
elif model_choice == 'SVC':
    # you can configure your model here
    clf_model = SVC(C=1.0, kernel='linear', probability=False)
    clf_model_str = 'SVC_C1_linear'
else:
    raise ValueError(f"Unknown model_choice: {model_choice}")

# then, choose your decoding parameters
random_state = 42
explained_variance = 0.90
balance_method = 'subsample'
normalize = 'true'
obs_axs = 0
chans_axs = 1
time_axs = -1

# time-windowed decoding parameters
window_size = 64  # window size in samples (e.g., 64 samples = 250 ms at 256 Hz)
step_size = 16    # step size in samples (e.g., 16 samples = 62.5 ms at 256 Hz)
sampling_rate = 256  # sampling rate of the data in Hz
first_time_point = -1.5  # the time in seconds of the first sample in the epoch
tails = 1  # 1 for one-tailed (e.g., accuracy > chance), 2 for two-tailed
n_shuffle_perms = 2  # how many times to shuffle labels and train decoder to make chance decoding results - this iterates over splits, so end up with n_shuffle_perms * n_splits for number of folds

# whether to do stats across fold, repeat, or bootstrap
unit_of_analysis = 'repeat'

# whether to store individual folds (true) or sum them within repeats (false)
folds_as_samples = True if unit_of_analysis == 'fold' else False

# percentile stats parameters
percentile = 95
cluster_percentile = 95

# additional parameters for permutation cluster stats
stat_func_choice = 'ttest_ind'  # 'ttest_ind', 'ttest_rel' or 'mean_diff'

if stat_func_choice == 'mean_diff':
    stat_func = mean_diff
    stat_func_str = 'mean_diff'
elif stat_func_choice == 'ttest_ind':
    stat_func = partial(ttest_ind, equal_var=False, nan_policy='omit')
    stat_func_str = 'ttest_ind'
elif stat_func_choice == 'ttest_rel':
    stat_func = partial(ttest_rel, nan_policy='omit')
    stat_func_str = 'ttest_rel'

p_thresh_for_time_perm_cluster_stats = 0.025
p_cluster = 0.025
permutation_type = 'independent'
# cluster_tails = 2

# plotting
single_column = True
show_legend = False
run_visualization_debug = False  # collapsed onto the first two PCs, this plots each trial and the SVM or LDA hyperplane.

condition_name = 'stimulus_congruency_by_switch_prop_block_balanced_conditions'
conditions = get_conditions_obj(condition_name)
condition_label = condition_name  # pass this into args for output path naming
condition_comparisons = get_comparisons(condition_name)

# epochs file selection - just use this one for testing
epochs_root_file = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

# which electrodes to use (all or sig) - TODO: add in an option to include a dictionary of electrodes here, like congruencySigElectrodes
electrodes = 'sig'

# # # # testing params (comment out)
subjects = ['D0103']
# subjects = ['D0057', 'D0059', 'D0063', 'D0069', 'D0071', 'D0077', 'D0090', 'D0094', 'D0100', 'D0102', 'D0103', 'D0107A', 'D0110', 'D0116', 'D0117', 'D0121', 'D0130', 'D0133', 'D0134']
n_splits = 2
n_repeats = 1
n_perm = 2
n_cluster_perms = 2
bootstraps = 2
n_jobs = -1
rois_dict = {
     'dlpfc': ["G_front_middle", "G_front_sup", "S_front_inf", "S_front_middle", "S_front_sup"],
}

balance_strata=True

args = SimpleNamespace(
    first_time_point = -1.5, # The time in seconds of the first sample in the epoch

    # Bootstrap control
    bootstraps=bootstraps,
    random_state=random_state,
    subjects=subjects,

    # Axes
    obs_axs=obs_axs,
    chans_axs=chans_axs,
    time_axs=time_axs,

    # Decoder / CV
    clf_model=clf_model,
    n_splits=n_splits,
    n_repeats=n_repeats,
    explained_variance=explained_variance,
    balance_method=balance_method,
    balance_strata=balance_strata,
    folds_as_samples=folds_as_samples,

    # Time-window decoding
    window_size=window_size,
    step_size=step_size,
    sampling_rate=sampling_rate,
    n_shuffle_perms=n_shuffle_perms,

    # Pooled shuffle
    conditions=conditions,  # the list from experiment_conditions.stimulus_lwpc_conditions

)

In [ ]:
conditions

{'Stimulus_c_MI_MR': {'BIDS_events': ['Stimulus/c75.0/r25.0',
   'Stimulus/c75.0/s25.0'],
  'metadata_query': "congruency == 'c' and incongruent_proportion == 75 and switch_proportion == 25"},
 'Stimulus_c_MI_MS': {'BIDS_events': ['Stimulus/c75.0/s75.0',
   'Stimulus/c75.0/r75.0'],
  'metadata_query': "congruency == 'c' and incongruent_proportion == 75 and switch_proportion == 75"},
 'Stimulus_c_MC_MR': {'BIDS_events': ['Stimulus/c25.0/r25.0',
   'Stimulus/c25.0/s25.0'],
  'metadata_query': "congruency == 'c' and incongruent_proportion == 25 and switch_proportion == 25"},
 'Stimulus_c_MC_MS': {'BIDS_events': ['Stimulus/c25.0/s75.0',
   'Stimulus/c25.0/r75.0'],
  'metadata_query': "congruency == 'c' and incongruent_proportion == 25 and switch_proportion == 75"},
 'Stimulus_i_MI_MR': {'BIDS_events': ['Stimulus/i75.0/r25.0',
   'Stimulus/i75.0/s25.0'],
  'metadata_query': "congruency == 'i' and incongruent_proportion == 75 and switch_proportion == 25"},
 'Stimulus_i_MI_MS': {'BIDS_events'

In [ ]:
condition_label

'stimulus_congruency_by_switch_prop_block_balanced_conditions'

In [ ]:
condition_comparisons

{'i_vs_c_at_sw25': [['Stimulus_i_MI_MR', 'Stimulus_i_MC_MR'],
  ['Stimulus_c_MI_MR', 'Stimulus_c_MC_MR']],
 'i_vs_c_at_sw75': [['Stimulus_i_MI_MS', 'Stimulus_i_MC_MS'],
  ['Stimulus_c_MI_MS', 'Stimulus_c_MC_MS']]}

set up paths

In [ ]:
# Determine LAB_root based on the operating system and environment
LAB_root = get_default_LAB_root()

print('LAB_root: ', LAB_root)

config_dir = os.path.join(project_root, 'src', 'analysis', 'config')
subjects_electrodestoROIs_dict = load_subjects_electrodes_to_ROIs_dict(save_dir=config_dir, filename='subjects_electrodestoROIs_dict.json')

condition_names = list(conditions.keys()) # get the condition names as a list

save_dir = os.path.join(current_script_dir, 'figs', f"{epochs_root_file}", 'sandbox')
os.makedirs(save_dir, exist_ok=True)
print(f"Save directory created or already exists at: {save_dir}")

LAB_root:  /cwork/jz421
Loaded data from /hpc/group/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json
Save directory created or already exists at: /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/sandbox


load data (here again, all variables normally have args in front of them but dropping that prefix for testing since those variables are defined in this notebook now rather than imported from the run script)

In [6]:
sig_chans_per_subject = get_sig_chans_per_subject(subjects, epochs_root_file, task=task, LAB_root=LAB_root)

rois = list(rois_dict.keys())
all_electrodes_per_subject_roi, sig_electrodes_per_subject_roi = make_sig_electrodes_per_subject_and_roi_dict(rois_dict, subjects_electrodestoROIs_dict, sig_chans_per_subject)
    
subjects_mne_objects = create_subjects_mne_objects_dict(subjects=subjects, epochs_root_file=epochs_root_file, conditions=conditions, task=task, just_HG_ev1_rescaled=True, acc_trials_only=acc_trials_only)

# determine which electrodes to use (all electrodes or just the task-significant ones)
if electrodes == 'all':
    raw_electrodes = all_electrodes_per_subject_roi 
    elec_string_to_add_to_filename = 'all_elecs'
elif electrodes == 'sig':
    raw_electrodes = sig_electrodes_per_subject_roi
    elec_string_to_add_to_filename = 'sig_elecs'

else:
    raise ValueError("electrodes input must be set to all or sig")

# ADD THIS BLOCK to create a string for the sampling method
folds_info_str = 'folds_as_samples' if folds_as_samples else 'repeats_as_samples'

# filter electrodes to only the ones that exist in the epochs objects. This mismatch can arise due to dropping channels when making the epochs objects, because the subjects_electrodestoROIs_dict is made based on all the electrodes, with no dropping.
electrodes = filter_electrode_lists_against_subjects_mne_objects(rois, raw_electrodes, subjects_mne_objects)

print_summary_of_dropped_electrodes(raw_electrodes, electrodes)

# get the confusion matrix using the downsampled version
# add elec and subject info to filename 6/11/25
other_string_to_add = (
    f"{elec_string_to_add_to_filename}_{str(len(subjects))}_subjects_{folds_info_str}_ev_{explained_variance}"
)

Loaded significant channels for subject D0103
For subject D0057, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['RAI12', 'RAI13', 'RAI14', 'RAI15', 'RAI16', 'RPI15', 'RPI14', 'RAMF10', 'RAMF11', 'RAMF12', 'RAMF13', 'RAMF14']
For subject D0059, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['LMMF9', 'LMMF11', 'LMMF10', 'LMMF12', 'LPSF16']
For subject D0063, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['LASF10', 'LASF11', 'LASF14', 'LASF15', 'LASF16', 'LMSF5', 'LMSF6', 'LMSF12', 'LPSF10', 'LPSF12', 'ROF16', 'RAI16', 'RAMF11', 'RAMF12', 'RAMF13', 'RAMF14', 'RMMF13', 'RMMF14', 'RAI10', 'RAI11', 'RASF15', 'RASF16', 'RMSF8', 'RMSF9', 'RMSF10', 'RMSF11', 'RMSF12', 'RMSF7', 'RAMF8', 'RAMF9', 'RAMF10', 'RMMF9', 'RMMF10']
For subject D0065, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['RASF13', 'RASF14', 'RASF15', 'RASF16', 'RMSF11', 

try loading data using metadata instead - throw this into general_utils.py once it works

decoding (hmm refactor this to use the metadata labels instead 4/24)

In [7]:
# use a unique random state for each bootstrap to ensure independent and reproducible results
bootstrap_random_state = 42

# this dictionary will store all results for this single bootstrap
results_for_this_bootstrap = {
    'time_window_results': {},
    'time_averaged_cms': {},
    'cats_by_roi': {}
}

roi_labeled_arrays_this_bootstrap_list = make_bootstrapped_roi_labeled_arrays_with_nan_trials_removed_for_each_channel(
    rois=rois,
    subjects_data_objects=subjects_mne_objects,
    condition_names=condition_names,
    subjects=args.subjects,
    electrodes_per_subject_roi=electrodes,
    n_bootstraps=1,  # Generate only one sample
    chans_axs=args.chans_axs,
    time_axs=args.time_axs,
    freq_axs=None,
    random_state=bootstrap_random_state, # Unique seed
    n_jobs=1  # Run ROI generation serially within this worker
)

# Extract the single LabeledArray from the list returned for each ROI
roi_labeled_arrays_this_bootstrap = {
    roi: arrays[0] for roi, arrays in roi_labeled_arrays_this_bootstrap_list.items() if arrays
}


Making LabeledArray for ROI: dlpfc with NaN trials removed within each channel
ROI dlpfc: Final conditions in nan_removed_data_dict: ['Stimulus_c_MI_MR', 'Stimulus_c_MI_MS', 'Stimulus_c_MC_MR', 'Stimulus_c_MC_MS', 'Stimulus_i_MI_MR', 'Stimulus_i_MI_MS', 'Stimulus_i_MC_MR', 'Stimulus_i_MC_MS']
condition 'Stimulus_c_MI_MR': subsampling to 23 trials
condition 'Stimulus_c_MI_MS': subsampling to 18 trials
condition 'Stimulus_c_MC_MR': subsampling to 82 trials
condition 'Stimulus_c_MC_MS': subsampling to 77 trials
condition 'Stimulus_i_MI_MR': subsampling to 68 trials
condition 'Stimulus_i_MI_MS': subsampling to 45 trials
condition 'Stimulus_i_MC_MR': subsampling to 25 trials
condition 'Stimulus_i_MC_MS': subsampling to 21 trials
generating 1 bootstrap sample(s)
Built bootstrapped_conditions_data with conditions: ['Stimulus_c_MI_MR', 'Stimulus_c_MI_MS', 'Stimulus_c_MC_MR', 'Stimulus_c_MC_MS', 'Stimulus_i_MI_MR', 'Stimulus_i_MI_MS', 'Stimulus_i_MC_MR', 'Stimulus_i_MC_MS']
Bootstrap 0: Labele

In [8]:
first_condition_comparison = next(iter(condition_comparisons))
strings_to_find = condition_comparisons[first_condition_comparison]
first_roi = next(iter(roi_labeled_arrays_this_bootstrap))
concatenated_data, labels, cats = concatenate_and_balance_data_for_decoding(
    roi_labeled_arrays_this_bootstrap, first_roi, strings_to_find, obs_axs, balance_method, balance_strata=balance_strata, random_state=random_state
)

X_flat = concatenated_data.reshape(concatenated_data.shape[0], -1)
y = labels

decoder = Decoder(
    cats,
    explained_variance=args.explained_variance,
    oversample=False,
    n_splits=2,
    n_repeats=1,
    clf=LinearDiscriminantAnalysis(),
    random_state=42
)

decoder.fit(X_flat, y)
pca = decoder.model.named_steps['pca']
n_kept = pca.n_components_
ratios = pca.explained_variance_ratio_
total = ratios.sum()
comps = pca.components_
mean_ = pca.mean_
print(f"[{first_roi} kept {n_kept} PCs, total var = {total:.1%}]")
print(f" per-PC: {np.round(ratios[:10], 3)}{' ...' if n_kept > 10 else ''}")
decoder.model.named_steps



ROI: dlpfc
  [NaN filter] class=0 stratum=Stimulus_i_MI_MR: 82 → 68 trials (17.1% dropped)
  [NaN filter] class=0 stratum=Stimulus_i_MC_MR: 82 → 25 trials (69.5% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MI_MR: 82 → 23 trials (72.0% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MC_MR: 82 → 82 trials (0.0% dropped)
  [balance_strata post-NaN] class=0: {'Stimulus_i_MI_MR': 68, 'Stimulus_i_MC_MR': 25} → 25 per stratum
  [balance_strata post-NaN] class=1: {'Stimulus_c_MI_MR': 23, 'Stimulus_c_MC_MR': 82} → 23 per stratum
  [class totals after stratum balancing] {0: 50, 1: 46}
  [balance_method=subsample] subsampling each class to 46
  [final] data shape: (92, 4, 641), labels: {np.int64(0): np.int64(46), np.int64(1): np.int64(46)}

No initial parameters
No initial parameters
[dlpfc kept 60 PCs, total var = 90.2%]
 per-PC: [0.038 0.038 0.036 0.032 0.029 0.028 0.027 0.025 0.025 0.024] ...


{'pca': PCA(n_components=0.9), 'model': LinearDiscriminantAnalysis()}

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from src.analysis.config import experiment_conditions
from src.analysis.config.condition_registry import get_comparisons, get_pooled_shuffle_settings, get_conditions_obj
from src.analysis.utils.labeled_array_utils import make_bootstrapped_roi_labeled_arrays_with_nan_trials_removed_for_each_channel
from src.analysis.decoding.decoding import concatenate_and_balance_data_for_decoding, plot_high_dim_decision_slice, plot_pca_3d_trajectory, plot_static_pca_projection, plot_pca_over_time


In [17]:
conditions = get_conditions_obj(condition_label)
condition_comparisons = get_comparisons(condition_label)
condition_names = list(conditions.keys()) # get the condition names as a list

print(f"\n{'='*20} 🔬 RUNNING 2D VISUALIZATION DEBUG (first two PCs and decision boundary) {'='*20}\n")

# 2. Get the single data sample for visualization
print("Generating LabeledArray data for visualization (n_bootstraps=1)...")
roi_labeled_arrays_viz = make_bootstrapped_roi_labeled_arrays_with_nan_trials_removed_for_each_channel(
    rois=rois,
    subjects_data_objects=subjects_mne_objects,
    condition_names=condition_names, # This is already defined in main()
    subjects=args.subjects,
    electrodes_per_subject_roi=electrodes, # This is already defined in main()
    n_bootstraps=1,
    chans_axs=args.chans_axs,
    time_axs=args.time_axs,
    random_state=args.random_state,
    n_jobs=1
)
roi_labeled_arrays_viz = {roi: arrs[0] for roi, arrs in roi_labeled_arrays_viz.items() if arrs}



==================== 🔬 RUNNING 2D VISUALIZATION DEBUG (first two PCs and decision boundary) ====================

Generating LabeledArray data for visualization (n_bootstraps=1)...

Making LabeledArray for ROI: dlpfc with NaN trials removed within each channel
ROI dlpfc: Final conditions in nan_removed_data_dict: ['Stimulus_c_MI_MR', 'Stimulus_c_MI_MS', 'Stimulus_c_MC_MR', 'Stimulus_c_MC_MS', 'Stimulus_i_MI_MR', 'Stimulus_i_MI_MS', 'Stimulus_i_MC_MR', 'Stimulus_i_MC_MS']
condition 'Stimulus_c_MI_MR': subsampling to 23 trials
condition 'Stimulus_c_MI_MS': subsampling to 18 trials
condition 'Stimulus_c_MC_MR': subsampling to 82 trials
condition 'Stimulus_c_MC_MS': subsampling to 77 trials
condition 'Stimulus_i_MI_MR': subsampling to 68 trials
condition 'Stimulus_i_MI_MS': subsampling to 45 trials
condition 'Stimulus_i_MC_MR': subsampling to 25 trials
condition 'Stimulus_i_MC_MS': subsampling to 21 trials
generating 1 bootstrap sample(s)
Built bootstrapped_conditions_data with conditions

In [34]:
# 3. Loop through ROIs and Pairs and plot
for condition_comparison, strings_to_find in condition_comparisons.items():

    for roi in rois:
        if roi not in roi_labeled_arrays_viz:
            print(f"Skipping visualization for {roi}: No data found.")
            continue
        
        pair_name = f"{strings_to_find[0][0]}_vs_{strings_to_find[1][0]}"
        print(f"\n--- Plotting for ROI: {roi}, Pair: {pair_name} ---")

        # 4. Get balanced data and 'cats'
        data, labels, cats = concatenate_and_balance_data_for_decoding(
            roi_labeled_arrays_viz, roi, strings_to_find, args.obs_axs,
            balance_method='subsample', # Must use subsample for this viz
            random_state=args.random_state
        )
        if data.size == 0:
            print("No data after balancing. Skipping plot.")
            continue
            
        data_flat = data.reshape(data.shape[0], -1)
    
        plot_static_pca_projection(
            roi_labeled_arrays_viz,
            roi,
            strings_to_find,
            cats,
            save_dir=os.path.join(save_dir, f"{condition_comparison}", f"{roi}"),
            obs_axs=args.obs_axs,
            random_state=args.random_state
        )
        
        plot_pca_over_time(
            roi_labeled_arrays_viz,
            roi,
            strings_to_find,
            cats,
            window_size=args.window_size,
            step_size=args.step_size,
            sampling_rate=args.sampling_rate,
            first_time_point=args.first_time_point,
            save_dir=os.path.join(save_dir, f"{condition_comparison}", f"{roi}"),
            obs_axs=args.obs_axs,
            random_state=args.random_state,
        )
        
        plot_pca_3d_trajectory(
            roi_labeled_arrays_viz, roi, strings_to_find, cats,
            save_dir=os.path.join(save_dir, f"{condition_comparison}", f"{roi}"),
            window_size=args.window_size,
            step_size=args.step_size,
            sampling_rate=args.sampling_rate,
            first_time_point=args.first_time_point,
            obs_axs=args.obs_axs,
            random_state=args.random_state,
            explained_variance=args.explained_variance,
            clf=args.clf_model
        )
                
print(f"\n{'='*20} ✅ VISUALIZATION DEBUG COMPLETE {'='*20}\n")


--- Plotting for ROI: dlpfc, Pair: Stimulus_i_MI_MR_vs_Stimulus_c_MI_MR ---

ROI: dlpfc
  [NaN filter] class=0 stratum=Stimulus_i_MI_MR: 82 → 68 trials (17.1% dropped)
  [NaN filter] class=0 stratum=Stimulus_i_MC_MR: 82 → 25 trials (69.5% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MI_MR: 82 → 23 trials (72.0% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MC_MR: 82 → 82 trials (0.0% dropped)
  [balance_strata post-NaN] class=0: {'Stimulus_i_MI_MR': 68, 'Stimulus_i_MC_MR': 25} → 25 per stratum
  [balance_strata post-NaN] class=1: {'Stimulus_c_MI_MR': 23, 'Stimulus_c_MC_MR': 82} → 23 per stratum
  [class totals after stratum balancing] {0: 50, 1: 46}
  [balance_method=subsample] subsampling each class to 46
  [final] data shape: (92, 4, 641), labels: {np.int64(0): np.int64(46), np.int64(1): np.int64(46)}

--- Generating Static PCA Plot for ROI: dlpfc ---

ROI: dlpfc
  [NaN filter] class=0 stratum=Stimulus_i_MI_MR: 82 → 68 trials (17.1% dropped)
  [NaN filter] class=0 stratum

/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto

Saved windowed PCA plot to /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/sandbox/i_vs_c_at_sw25/dlpfc/pca_over_time_dlpfc.pdf

ROI: dlpfc
  [NaN filter] class=0 stratum=Stimulus_i_MI_MR: 82 → 68 trials (17.1% dropped)
  [NaN filter] class=0 stratum=Stimulus_i_MC_MR: 82 → 25 trials (69.5% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MI_MR: 82 → 23 trials (72.0% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MC_MR: 82 → 82 trials (0.0% dropped)
  [balance_strata post-NaN] class=0: {'Stimulus_i_MI_MR': 68, 'Stimulus_i_MC_MR': 25} → 25 per stratum
  [balance_strata post-NaN] class=1: {'Stimulus_c_MI_MR': 23, 'Stimulus_c_MC_MR': 82} → 23 per stratum
  [class totals after stratum balancing] {0: 50, 1: 46}
  [balance_method=subsample] subsampling each class to 46
  [final] data shape: (92, 

/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/sklearnex/decomposition/pca.py:347: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto

Saved windowed PCA plot to /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/sandbox/i_vs_c_at_sw75/dlpfc/pca_over_time_dlpfc.pdf

ROI: dlpfc
  [NaN filter] class=0 stratum=Stimulus_i_MI_MS: 82 → 45 trials (45.1% dropped)
  [NaN filter] class=0 stratum=Stimulus_i_MC_MS: 82 → 21 trials (74.4% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MI_MS: 82 → 18 trials (78.0% dropped)
  [NaN filter] class=1 stratum=Stimulus_c_MC_MS: 82 → 77 trials (6.1% dropped)
  [balance_strata post-NaN] class=0: {'Stimulus_i_MI_MS': 45, 'Stimulus_i_MC_MS': 21} → 21 per stratum
  [balance_strata post-NaN] class=1: {'Stimulus_c_MI_MS': 18, 'Stimulus_c_MC_MS': 77} → 18 per stratum
  [class totals after stratum balancing] {0: 42, 1: 36}
  [balance_method=subsample] subsampling each class to 36
  [final] data shape: (72, 

In [ ]:
# --- 1. Calculate Time-Averaged CMs (using raw counts) ---
for condition_comparison, strings_to_find in condition_comparisons.items():
    results_for_this_bootstrap['time_averaged_cms'][condition_comparison] = {}
    for roi in rois:
        # update this.
        # Get the 'cats' dictionary for this ROI - pass through balance_strata and bootstrap_random_state 
        _, cats = gather_class_data_by_stratum(
            roi_labeled_arrays_this_bootstrap, roi, strings_to_find, args.obs_axs
        )
        
        ## FIX: Store the 'cats' dictionary for this ROI so it can be used for plotting later.
        if roi not in results_for_this_bootstrap['cats_by_roi']:
            results_for_this_bootstrap['cats_by_roi'][roi] = cats
            
        # Get the raw-count confusion matrix
        cm = get_time_averaged_confusion_matrix(
            roi_labeled_arrays=roi_labeled_arrays_this_bootstrap,
            roi=roi,
            strings_to_find=strings_to_find,
            cats=cats,
            clf=args.clf_model,
            n_splits=args.n_splits,
            n_repeats=args.n_repeats,
            obs_axs=args.obs_axs,
            balance_method=args.balance_method,
            explained_variance=args.explained_variance,
            random_state=args.random_state,
        )
        if cm is not None:
            results_for_this_bootstrap['time_averaged_cms'][condition_comparison][roi] = cm

for condition_comparison, strings_to_find in condition_comparisons.items():
    
    results_for_this_bootstrap['time_window_results'][condition_comparison] = {}
    
    # Get confusion matrices for each ROI
    cm_true_per_roi, cm_shuffle_per_roi = get_confusion_matrices_for_rois_time_window_decoding_jim(
        roi_labeled_arrays=roi_labeled_arrays_this_bootstrap,
        rois=rois,
        condition_comparison=condition_comparison,
        strings_to_find=strings_to_find,
        n_splits=args.n_splits,
        n_repeats=args.n_repeats,
        obs_axs=args.obs_axs,
        time_axs=-1,
        clf=args.clf_model,
        balance_method=args.balance_method,
        explained_variance=args.explained_variance,
        random_state=bootstrap_random_state,
        window_size=args.window_size,
        step_size=args.step_size,
        n_perm=args.n_shuffle_perms,
        sampling_rate=args.sampling_rate,
        first_time_point=-1,
        folds_as_samples=args.folds_as_samples
    )

    for roi in rois:
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi] = {}
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['strings_to_find'] = strings_to_find

        cm_true = cm_true_per_roi[roi]['cm_true']
        cm_shuffle = cm_shuffle_per_roi[roi]['cm_shuffle']
        time_window_centers = cm_true_per_roi[roi]['time_window_centers']
        window_size = cm_true_per_roi[roi]['window_size']
        step_size = cm_true_per_roi[roi]['step_size']

        # store cm outputs and windowing parameters
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['cm_true'] = cm_true
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['cm_shuffle'] = cm_shuffle
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['time_window_centers'] = time_window_centers
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['window_size'] = window_size
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['step_size'] = step_size
        
        # Compute accuracies
        accuracies_true, accuracies_shuffle = compute_accuracies(cm_true, cm_shuffle)

        # find average accuracies, either across folds or repeats/perms, depending on whether folds_as_samples is set to true or not
        mean_accuracies_true = np.mean(accuracies_true, axis=1, keepdims=True) # Shape: (n_windows, 1), keeping folds/repeats dimension for compatibility with the percentile function
        mean_accuracies_shuffle = np.mean(accuracies_shuffle, axis=1, keepdims=True) 
        
        # store accuracies - TODO: average across bootstraps somehow for these
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['accuracies_true'] = accuracies_true
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['accuracies_shuffle'] = accuracies_shuffle
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['mean_accuracies_true'] = mean_accuracies_true
        results_for_this_bootstrap['time_window_results'][condition_comparison][roi]['mean_accuracies_shuffle'] = mean_accuracies_shuffle
  

 [balance_strata] class=['Stimulus_i_MI_MR', 'Stimulus_i_MC_MR']: sizes={'Stimulus_i_MI_MR': 82, 'Stimulus_i_MC_MR': 82}, subsampling each to 82
 [balance_strata] class=['Stimulus_c_MI_MR', 'Stimulus_c_MC_MR']: sizes={'Stimulus_c_MI_MR': 82, 'Stimulus_c_MC_MR': 82}, subsampling each to 82
The code is searching for strings: [['Stimulus_i_MI_MR', 'Stimulus_i_MC_MR'], ['Stimulus_c_MI_MR', 'Stimulus_c_MC_MR']]

 [balance_strata] class=['Stimulus_i_MI_MR', 'Stimulus_i_MC_MR']: sizes={'Stimulus_i_MI_MR': 82, 'Stimulus_i_MC_MR': 82}, subsampling each to 82
 [balance_strata] class=['Stimulus_c_MI_MR', 'Stimulus_c_MC_MR']: sizes={'Stimulus_c_MI_MR': 82, 'Stimulus_c_MC_MR': 82}, subsampling each to 82

ROI: dlpfc
Initial concatenated data shape: (328, 4, 641)
  Trials: 328
  Channels: 4
  Time points: 641

--- Detailed NaN Analysis ---

Missing Channels (all NaN across all trials): 0/4

Trial-level NaN Statistics:
  Trials with ANY NaN: 130/328 (39.6%)
  Trials with outlier NaNs (sparse): 0/328 